# RAG Structured Output
LLM JSON Schema를 원하는 구조로 응답하도록 처리할 수 있다.

`llm.with_structured_output(PydanticModelClass)`

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [2]:
# 가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query=None):
    return [
        Document(page_content="파리는 프랑스의 수도로, 연간 관광객 수 약 3000만 명입니다. 주요 관광지로는 센강, 개선문, 루브르 박물관이 있습니다."),  # 파리 문서
        Document(page_content="런던은 영국의 수도로, 연간 관광객 수 약 2000만 명입니다. 주요 관광지로는 버킹엄 궁전, 런던 아이, 타워 브릿지가 있습니다."),  # 런던 문서
        Document(page_content="교토는 일본의 옛 수도로, 연간 관광객 수 약 1500만 명입니다. 주요 관광지로는 금각사, 은각사, 기요미즈데라가 있습니다.")  # 교토 문서
    ]
retrieve_vectordb()


[Document(metadata={}, page_content='파리는 프랑스의 수도로, 연간 관광객 수 약 3000만 명입니다. 주요 관광지로는 센강, 개선문, 루브르 박물관이 있습니다.'),
 Document(metadata={}, page_content='런던은 영국의 수도로, 연간 관광객 수 약 2000만 명입니다. 주요 관광지로는 버킹엄 궁전, 런던 아이, 타워 브릿지가 있습니다.'),
 Document(metadata={}, page_content='교토는 일본의 옛 수도로, 연간 관광객 수 약 1500만 명입니다. 주요 관광지로는 금각사, 은각사, 기요미즈데라가 있습니다.')]

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field # 출력 스키마 검증/구조화
from typing import List # 리스트 타입 힌트

class CityInfo(BaseModel):
    city: str = Field(description='도시 이름')
    visitors: int = Field(description='연간 방문객 수')
    landmarks: list[str] = Field(description='주요 관광지 목록')

class CityList(BaseModel):
    cities: List[CityInfo]

llm = init_chat_model('gpt-5.6-luna')
prompt = ChatPromptTemplate.from_template('''
당신은 조회된 문서에서 사용자 원하는 정보를 추출하는 에이젼트입니다.
제공된 문서를 바탕으로 JSON형식으로 답변해주세요.

[조회된 문서]
{docs}

[사용자 질문]
{query}
''')

chain = prompt | llm.with_structured_output(CityList) # CityList 형으로 output을 강제

context = '\n\n'.join([doc.page_content for doc in retrieve_vectordb()])
query = '교토의 주요 관광 정보'
response: CityList = chain.invoke({'docs': context, 'query': query})

print(response)

cities=[CityInfo(city='교토', visitors=15000000, landmarks=['금각사', '은각사', '기요미즈데라'])]


In [5]:
query = '교토와 파리의 주요 관광 정보는?'
response: CityList = chain.invoke({'docs': context, 'query': query})

print(response)

cities=[CityInfo(city='교토', visitors=15000000, landmarks=['금각사', '은각사', '기요미즈데라']), CityInfo(city='파리', visitors=30000000, landmarks=['센강', '개선문', '루브르 박물관'])]


### Pydantic 문법
- Pydantic BaseModel로 스키마 정의 / 데이터 검증 / 자동 타입변환

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict, Annotated # 타입 힌트
from datetime import datetime

# Pydantic 모델 클래스 : 사용자 데이터를 검증/파싱
class User(BaseModel):
    id: int
    name: str
    email: str = None # 기본값 None
    is_activate: bool = True # 기본값 True
    created_at: datetime

honggd = User(
    id = 1,
    name = '홍길동',
    email = 'gd@hong.com',
    is_activate=True,
    created_at=datetime.now()
)

print(honggd)

id=1 name='홍길동' email='gd@hong.com' is_activate=True created_at=datetime.datetime(2026, 9, 2, 16, 8, 1, 565394)


In [ ]:
# id는 문자형 -> 숫자형 자동파싱, 기본값(옵션)은 기본설정
sinsa = User(id='2', name='신사임당',created_at=datetime.now())
print(sinsa)

id=2 name='신사임당' email=None is_activate=True created_at=datetime.datetime(2026, 9, 2, 16, 11, 56, 151711)


In [ ]:
class Product(BaseModel):
    code: str = Field(..., description='제품 식별 코드') # ... : 필수값
    name: str = Field(..., description='제품명', min_length=1, max_length=10)
    price: float = Field(..., description='제품 가격', ge=0)
    quantity: int = Field(default=0, description='제품 수량')
    discription: str = Field(..., description='제품 설명', max_length=500)
    discount_rate: float = Field(default=0.0, description='제품 할인율', ge=0, le=1)
    created_at: datetime = Field(default=datetime.now(), description='제품 등록일')

In [ ]:
# 형태가 달라도 자동 형변환 가능시 자동 파싱 / 필수값이 아니면 생략 가능 (기본값 자동 설정)
prod = Product(
    code = 'x01',
    name = '머그컵',
    price = 2000,
    quantity = 85,
    discription = '소 그림이 그려져있는 머그컵',
    discount_rate = 0.1
)

prod

Product(code='x01', name='머그컵', price=2000.0, quantity=85, discription='소 그림이 그려져있는 머그컵', discount_rate=0.1, created_at=datetime.datetime(2026, 9, 2, 16, 17, 44, 742959))

In [ ]:
# Pydantic 객체 -> JSON으로 변환
prod_json_str = prod.model_dump_json()
print(prod_json_str)


{"code":"x01","name":"머그컵","price":2000.0,"quantity":85,"discription":"소 그림이 그려져있는 머그컵","discount_rate":0.1,"created_at":"2026-09-02T16:17:44.742959"}


In [20]:
# Pydantic 객체 -> dict로 변환
prod_dict = prod.model_dump()
print(prod_dict)

{'code': 'x01', 'name': '머그컵', 'price': 2000.0, 'quantity': 85, 'discription': '소 그림이 그려져있는 머그컵', 'discount_rate': 0.1, 'created_at': datetime.datetime(2026, 9, 2, 16, 17, 44, 742959)}


In [ ]:
# json -> Pydantic 객체로 변환
json_str = '{"code":"x01","name":"머그컵","price":2000.0,"quantity":85,"discription":"소 그림이 그려져있는 머그컵","discount_rate":0.1,"created_at":"2026-09-02T16:17:44.742959"}'
prod_from_json = Product.model_validate_json(json_str) # JSON -> Pydantic 검증/파싱
print(prod_from_json)
print(type(prod_from_json))

code='x01' name='머그컵' price=2000.0 quantity=85 discription='소 그림이 그려져있는 머그컵' discount_rate=0.1 created_at=datetime.datetime(2026, 9, 2, 16, 17, 44, 742959)
<class '__main__.Product'>


In [28]:
# dict -> Pydantic 객체로 변환

data = {'code': 'x01', 'name': '머그컵', 'price': 2000.0, 'quantity': 85, 'discription': '소 그림이 그려져있는 머그컵', 'discount_rate': 0.1, 'created_at': datetime(2026, 9, 2, 16, 17, 44, 742959)}

prod_from_dict = Product(**data)
print(prod_from_dict)
print(type(prod_from_dict))


code='x01' name='머그컵' price=2000.0 quantity=85 discription='소 그림이 그려져있는 머그컵' discount_rate=0.1 created_at=datetime.datetime(2026, 9, 2, 16, 17, 44, 742959)
<class '__main__.Product'>


## 실습 정리 (RAG + Structured Output)

- 본 실습에서는 RAG 파이프라인에서 LLM의 응답을 **Structured Output(JSON)** 형태로 받는 방법을 다루었다.
- 조회된 문서를 기반으로 답변을 생성하되, `Pydantic(BaseModel)` 스키마를 통해  
  **응답 형식과 필드 구조를 강제**함으로써 결과의 일관성과 신뢰성을 확보했다.
- Structured Output을 적용하면  
  - 필요한 정보만 정확히 추출  
  - 불필요한 자연어 응답 최소화  
  - 후처리 코드 단순화  
  가 가능하다.
- 이는 RAG를 **검색 → 생성 → 응답 표시** 수준이 아닌,  
  **검색 → 정보 추출 → 시스템 연계** 단계로 확장할 수 있게 해준다.
- 실무에서는 Structured Output 기반 RAG가  
  - 사내 문서 정보 추출  
  - 고객 응답 데이터 정형화  
  - API/DB 연동 자동화  
  등에 활용되며, 운영 가능한 서비스 형태의 RAG 구현에 필수적인 패턴이다.